# Declarative Pipelines
## Build a bronze → silver → gold medallion by *declaring* tables in SQL.

*Running on open-source **Apache Spark 4.1** + **Unity Catalog OSS** — no proprietary runtime.*

In [ ]:
# Run me first — resets to a clean slate and starts Spark (~10s).
import os, subprocess
from pyspark.sql import SparkSession
from IPython.display import Code
NS = os.environ.get('DEMO_NS','medallion_demo').rstrip('_') or 'medallion_demo'
subprocess.run(['python3','/home/jovyan/demos/sdp-medallion/reset.py'], capture_output=True)
spark = SparkSession.builder.getOrCreate()
print('Ready · your tables land in  managed_demo.' + NS)

## A pipeline is just SQL
Each table in this medallion is **one SQL `SELECT`**. There's no orchestration code and no write statements — you declare *what* the table is, and Spark builds, orders, and writes the rest.

In [ ]:
!ls -1 /home/jovyan/demos/sdp-medallion/transformations

The **silver** layer below parses the raw order JSON, adds time features, and joins the city dimension. Spark reads the table names in the `FROM`/`JOIN` and wires the dependency graph automatically — you never write the order of operations.

In [ ]:
Code(filename='/home/jovyan/demos/sdp-medallion/transformations/silver_orders_enriched.sql')

## Spark validates the whole pipeline *before* it runs
A **dry-run** checks every table reference up front — no data moves. Here's a version of the pipeline with a typo (`orderz` instead of `orders`):

In [ ]:
Code(filename='/home/jovyan/demos/sdp-medallion/broken/transformations/gold_revenue.sql')

In [ ]:
!spark-pipelines dry-run --spec /home/jovyan/demos/sdp-medallion/broken/spark-pipeline.yml 2>&1 | grep -iE 'NOT_FOUND|Failed to resolve|orderz' || true

Caught — the exact table and line, **before a single byte moved**. The kind of bug that would otherwise crash a job halfway through.

## Run the real pipeline
Spark runs bronze → silver → gold in dependency order and registers Delta tables in the catalog:

In [ ]:
!spark-pipelines run --spec /home/jovyan/demos/sdp-medallion/spark-pipeline.yml 2>&1 | grep -E 'is COMPLETED|Run is COMPLETED'

## The result — revenue per brand, straight from the catalog

In [ ]:
bs = spark.table(f'managed_demo.{NS}.gold_brand_summary').orderBy('total_revenue', ascending=False).toPandas()
bs

In [ ]:
import matplotlib.pyplot as plt
top = bs.head(10)[::-1]
plt.figure(figsize=(8,4)); plt.barh(top['brand_name'], top['total_revenue'])
plt.xlabel('revenue ($)'); plt.title('Revenue by brand'); plt.tight_layout(); plt.show()

### What just happened
Plain SQL `SELECT`s became a real bronze → silver → gold pipeline: Spark inferred the dependency graph, ran the layers in order, caught a bad reference before it cost a job, and registered the tables in the catalog — all on **open-source Spark and Unity Catalog**.